# Metal Gear Agent: Solutions Notebook

This notebook contains complete solutions for both TODOs.

In [ ]:
# ── Setup (run this first) ──────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/metal_gear_agent")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")

In [ ]:
from gear_agent import *
from gear_agent.data import CellType, NPC, NPC_CATALOG
from gear_agent.game import OperativeState
from gear_agent.scenario import FacilityWorld
from gear_agent.agent import TOOLS_DESCRIPTION, parse_tool_call
from gear_agent.oracle import ORACLE_TEMPLATE, llm_oracle
from gear_agent.display import render_grid
print("Metal Gear Agent loaded!")

In [ ]:
# Show the full facility map
operative, world, tools = create_game()
print(render_grid(world, operative, show_all=True))

---
## Play the Game Yourself!

Before looking at the AI solutions, try playing the game manually to understand the facility layout, the missions, and the challenge. Use the buttons to move, contact NPCs via codec, collect items, and try to gather 10 intel.

In [ ]:
from gear_agent.interactive import play_interactive

game = play_interactive()

---
## Solution: TODO 1 — Rule-Based Think Function

Strategy: systematic exploration with priority-based actions.
- Collect items on current cell
- Contact NPCs/safe room operators/armory technicians
- Equip/fabricate when we have the materials
- Engage bosses when we have the right equipment
- Move toward unvisited cells, preferring cells with interesting content

In [ ]:
from collections import deque

def _bfs_next_step(start, target, world):
    """BFS pathfinding that navigates around reinforced walls."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) == target:
            return path[0] if path else None
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def think_rule_based(operative: OperativeState, world: FacilityWorld, history: list[dict]) -> str:
    row, col = operative.position
    cell = world.get_cell(row, col)

    # --- Priority 1: Collect items on current cell ---
    if cell.items:
        return 'TOOL: collect()'

    # --- Priority 2: Interact with current cell ---
    if cell.cell_type == CellType.INFORMANT and cell.npc:
        npc_name = cell.npc.name
        talked = sum(1 for e in operative.briefing_log if npc_name in e)
        if talked == 0:
            return 'TOOL: codec(question="What can you tell me? What missions or dangers should I know about?")'
        if cell.npc_id == "the_engineer" and not operative.has_item("Encrypted Message") and not world.errand_message_picked_up:
            return 'TOOL: codec(question="Do you have any message or transmission job for me?")'
        if cell.npc_id == "the_informant" and operative.has_item("Detonator Components"):
            return 'TOOL: codec(question="I have detonator components. Can you assemble explosives?")'
        if cell.npc_id == "the_colonel" and operative.has_item("Encrypted Message"):
            return 'TOOL: codec(question="I have an encrypted message to deliver to you.")'

    if cell.cell_type == CellType.SAFE_ROOM:
        if operative.position == (2, 5):
            if operative.has_item("Medical Supplies"):
                return 'TOOL: codec(question="I brought medical supplies from the south!")'
            if operative.has_item("Plastic Explosives"):
                return 'TOOL: codec(question="I have plastic explosives for the demolition op!")'
        if operative.position == (7, 4) and not world.errand_medical_picked_up:
            return 'TOOL: codec(question="Hello, do you need any help?")'
        if operative.health < operative.MAX_HEALTH and operative.intel >= 2:
            return 'TOOL: hide()'

    if cell.cell_type == CellType.ARMORY:
        if operative.has_item("Circuit Board") and operative.intel >= 1 and operative.position == (6, 2):
            return 'TOOL: equip(item="EMP Device")'
        if operative.has_item("C4 Compound") and operative.intel >= 1 and operative.position == (4, 1):
            return 'TOOL: equip(item="C4 Package")'
        if operative.has_item("Access Codes") and operative.position == (4, 1):
            return 'TOOL: equip(item="sell Access Codes")'
        if operative.has_item("Hostage Data") and operative.position == (4, 1):
            return 'TOOL: equip(item="sell Hostage Data")'

    if cell.cell_type == CellType.BOSS_ARENA:
        if cell.boss_name == "Security Mainframe" and operative.has_item("EMP Device") and world.mainframe_active:
            return 'TOOL: engage()'
        if cell.boss_name == "Armored Mech" and operative.has_item("C4 Package") and world.mech_active:
            return 'TOOL: engage()'

    if operative.health <= 1:
        if operative.has_item("Medkit"):
            return 'TOOL: use(item="Medkit")'
        if operative.has_item("Ration"):
            return 'TOOL: use(item="Ration")'

    # --- Priority 3: Navigate to targets using BFS ---
    targets = []

    if operative.has_item("Encrypted Message"):
        targets.append((7, 1))
    if operative.has_item("Medical Supplies"):
        targets.append((2, 5))
    if operative.has_item("Plastic Explosives"):
        targets.append((2, 5))
    if operative.has_item("Circuit Board") and not operative.has_item("EMP Device"):
        targets.append((6, 2))
    if operative.has_item("EMP Device") and world.mainframe_active:
        targets.append((2, 1))
    if operative.has_item("C4 Compound") and not operative.has_item("C4 Package"):
        targets.append((4, 1))
    if operative.has_item("C4 Package") and world.mech_active:
        targets.append((4, 6))
    if operative.has_item("Access Codes") or operative.has_item("Hostage Data"):
        targets.append((4, 1))

    for tc in [(5, 0), (6, 6), (0, 7)]:
        if tc not in operative.visited:
            targets.append(tc)

    if (7, 1) not in operative.visited:
        targets.append((7, 1))
    if (7, 4) not in operative.visited:
        targets.append((7, 4))
    if (5, 3) not in operative.visited:
        targets.append((5, 3))
    if (3, 4) not in operative.visited:
        targets.append((3, 4))

    if not operative.has_item("Circuit Board") and not operative.has_item("EMP Device"):
        if (2, 0) not in operative.visited:
            targets.append((2, 0))

    if not operative.has_item("Detonator Components") and not world.explosives_received:
        if (0, 2) not in operative.visited:
            targets.append((0, 2))

    seen = set()
    unique = []
    for t in targets:
        if t not in seen and t != (row, col):
            seen.add(t)
            unique.append(t)

    for target in unique:
        d = _bfs_next_step((row, col), target, world)
        if d:
            return f'TOOL: move(direction="{d}")'

    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if (nr, nc) not in operative.visited and world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'

    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'

    return 'TOOL: scan()'

In [ ]:
result = play_rule_based(think_rule_based, max_turns=100, delay=0.05)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'} | Intel: {result['intel']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

---
## Solution: TODO 2 — LLM Think Function

In [ ]:
import os
from google import genai

# os.environ["GEMINI_API_KEY"] = "your-key-here"
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

In [ ]:
import re
from collections import deque

# ── Helper: BFS pathfinding ──────────────────────────────────────────────────

def _bfs_next_step_llm(start, target, world):
    """BFS pathfinding that navigates around reinforced walls."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) == target:
            return path[0] if path else None
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def _bfs_nearest_unvisited(start, operative, world):
    """BFS to find direction toward the nearest unvisited passable cell."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) not in operative.visited and path:
            return path[0]
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def _get_quest_targets(operative, world):
    """Return a list of (row, col) targets based on current active missions.

    These are derived from what the agent has discovered through gameplay —
    items in inventory imply missions that were triggered by NPC conversations.
    """
    targets = []
    if operative.has_item("Encrypted Message"):
        targets.append((7, 1))  # The Colonel
    if operative.has_item("Medical Supplies"):
        targets.append((2, 5))  # North Safe Room
    if operative.has_item("Plastic Explosives"):
        targets.append((2, 5))  # North Safe Room
    if operative.has_item("Circuit Board") and not operative.has_item("EMP Device"):
        targets.append((6, 2))  # Weapons Lab
    if operative.has_item("EMP Device") and world.mainframe_active:
        targets.append((2, 1))  # Security Mainframe
    if operative.has_item("C4 Compound") and not operative.has_item("C4 Package"):
        targets.append((4, 1))  # Demolitions Workshop
    if operative.has_item("C4 Package") and world.mech_active:
        targets.append((4, 6))  # Armored Mech
    if operative.has_item("Access Codes") or operative.has_item("Hostage Data"):
        targets.append((4, 1))  # Demolitions Workshop
    return targets


def _bfs_navigate(operative, world):
    """BFS navigation toward quest targets, then nearest unvisited cell.

    This is the key architectural insight: navigation is a deterministic problem
    that LLMs are bad at. BFS always finds the shortest path. We let the LLM
    handle decisions (what to say, which mission to pursue) but never ask it
    to figure out cardinal directions from grid coordinates.
    """
    row, col = operative.position

    # Priority 1: BFS toward active mission targets
    for target in _get_quest_targets(operative, world):
        d = _bfs_next_step_llm((row, col), target, world)
        if d:
            return f'TOOL: move(direction="{d}")'

    # Priority 2: BFS to nearest unvisited cell (exploration)
    d = _bfs_nearest_unvisited((row, col), operative, world)
    if d:
        return f'TOOL: move(direction="{d}")'

    # Priority 3: any passable direction
    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'
    return 'TOOL: move(direction="north")'


# ── Agentic AI Pattern 1: Autopilot for Obvious Interactions ────────────────
# Before consulting the LLM, check if the current cell has an obvious action.
# This prevents the LLM from "walking past" NPCs, intel caches, and armories.
# It only fires when the agent is already standing on the right cell.

def _has_new_business_with_cell(operative, world, cell):
    """Check if the operative has something new to do on this cell."""
    if cell.items:
        return True
    if cell.cell_type == CellType.BOSS_ARENA:
        if cell.boss_name == "Security Mainframe" and operative.has_item("EMP Device") and world.mainframe_active:
            return True
        if cell.boss_name == "Armored Mech" and operative.has_item("C4 Package") and world.mech_active:
            return True
        return False
    if cell.cell_type == CellType.SAFE_ROOM:
        row, col = operative.position
        if (row, col) == (2, 5) and (operative.has_item("Medical Supplies") or operative.has_item("Plastic Explosives")):
            return True
        if (row, col) == (7, 4) and not world.errand_medical_picked_up:
            return True
        return False
    if cell.cell_type == CellType.ARMORY:
        row, col = operative.position
        if (row, col) == (6, 2) and operative.has_item("Circuit Board") and operative.intel >= 1:
            return True
        if (row, col) == (4, 1) and operative.has_item("C4 Compound") and operative.intel >= 1:
            return True
        if (row, col) == (4, 1) and (operative.has_item("Access Codes") or operative.has_item("Hostage Data")):
            return True
        return False
    if cell.cell_type == CellType.INFORMANT and cell.npc:
        if cell.npc_id == "the_engineer" and not world.errand_message_picked_up and not operative.has_item("Encrypted Message"):
            return True
        if cell.npc_id == "the_informant" and operative.has_item("Detonator Components"):
            return True
        if cell.npc_id == "the_colonel" and operative.has_item("Encrypted Message"):
            return True
        npc_name = cell.npc.name
        talked = any(npc_name in e for e in operative.briefing_log)
        if not talked:
            return True
        return False
    return False


def _auto_interact(operative, world, cell):
    """Return an action string for obvious interactions, or None to defer.

    This handles the WHAT — collecting items, contacting NPCs with the right
    keywords, equipping at armories, and engaging bosses.
    """
    # Always collect items on the current cell
    if cell.items:
        return 'TOOL: collect()'

    if not _has_new_business_with_cell(operative, world, cell):
        return None

    if cell.cell_type == CellType.SAFE_ROOM:
        row, col = operative.position
        if (row, col) == (2, 5) and operative.has_item("Medical Supplies"):
            return 'TOOL: codec(question="I brought medical supplies from the south!")'
        if (row, col) == (2, 5) and operative.has_item("Plastic Explosives"):
            return 'TOOL: codec(question="I have plastic explosives for the demolition op!")'
        if (row, col) == (7, 4) and not world.errand_medical_picked_up:
            return 'TOOL: codec(question="Reporting in. Do you need any help?")'
        return None

    if cell.cell_type == CellType.INFORMANT and cell.npc:
        # Engineer ALWAYS gets the keyword question (even on first visit)
        if cell.npc_id == "the_engineer":
            return 'TOOL: codec(question="Do you have a mission or errand for me?")'
        if cell.npc_id == "the_informant" and operative.has_item("Detonator Components"):
            return 'TOOL: codec(question="I have detonator components. Can you assemble explosives?")'
        if cell.npc_id == "the_colonel" and operative.has_item("Encrypted Message"):
            return 'TOOL: codec(question="I have an encrypted message to deliver to you.")'
        # First-time NPC encounter — ask for general info
        return 'TOOL: codec(question="What can you tell me about this facility?")'

    if cell.cell_type == CellType.ARMORY:
        row, col = operative.position
        if (row, col) == (6, 2) and operative.has_item("Circuit Board") and operative.intel >= 1:
            return 'TOOL: equip(item="EMP Device")'
        if (row, col) == (4, 1) and operative.has_item("C4 Compound") and operative.intel >= 1:
            return 'TOOL: equip(item="C4 Package")'
        if (row, col) == (4, 1) and operative.has_item("Access Codes"):
            return 'TOOL: equip(item="sell Access Codes")'
        if (row, col) == (4, 1) and operative.has_item("Hostage Data"):
            return 'TOOL: equip(item="sell Hostage Data")'
        return None

    if cell.cell_type == CellType.BOSS_ARENA:
        return 'TOOL: engage()'

    return None


# ── Agentic AI Pattern 2: Memory Management ─────────────────────────────────
# Instead of dumping raw history, we build a structured memory that summarizes
# what the agent has accomplished, what failed, and what it knows about the facility.

def _build_memory_summary(operative: OperativeState, world: FacilityWorld, history: list[dict], turns_used: int) -> str:
    """Build a structured memory summary from game history."""
    sections = []

    # -- Active missions FIRST --
    missions = []
    if operative.has_item("Encrypted Message"):
        missions.append("URGENT: Deliver Encrypted Message to The Colonel at (7,1) — mention 'message' or 'deliver' — reward: 2 intel")
    if operative.has_item("Medical Supplies"):
        missions.append("URGENT: Deliver Medical Supplies to North Safe Room at (2,5) — just codec() — reward: 2 intel")
    if operative.has_item("Detonator Components"):
        missions.append("URGENT: Bring Detonator Components to The Informant at (3,4) — mention 'detonator' or 'assemble' — reward: Plastic Explosives")
    if operative.has_item("Plastic Explosives"):
        missions.append("URGENT: Bring Plastic Explosives to North Safe Room at (2,5) — just codec() — reward: C4 Compound")
    if operative.has_item("Circuit Board") and not operative.has_item("EMP Device"):
        missions.append('URGENT: Fabricate EMP Device at Weapons Lab (6,2) — equip(item="EMP Device") — costs 1 intel')
    if operative.has_item("C4 Compound") and not operative.has_item("C4 Package"):
        missions.append('URGENT: Fabricate C4 Package at Demolitions Workshop (4,1) — equip(item="C4 Package") — costs 1 intel')
    if operative.has_item("EMP Device") and world.mainframe_active:
        missions.append("URGENT: Engage Security Mainframe at (2,1) with EMP Device — engage() — reward: 3 intel + Access Codes")
    if operative.has_item("C4 Package") and world.mech_active:
        missions.append("URGENT: Engage Armored Mech at (4,6) with C4 Package — engage() — reward: 3 intel + Hostage Data")
    if operative.has_item("Access Codes"):
        missions.append('Trade Access Codes at Demolitions Workshop (4,1) — equip(item="sell Access Codes") — reward: 1 intel')
    if operative.has_item("Hostage Data"):
        missions.append('Trade Hostage Data at Demolitions Workshop (4,1) — equip(item="sell Hostage Data") — reward: 1 intel')
    if missions:
        sections.append("=== ACTIVE MISSIONS (do these NOW) ===\n" + "\n".join(f"  >>> {m}" for m in missions))

    # -- Urgency warning --
    turns_left = 100 - turns_used
    intel_needed = 10 - operative.intel
    if turns_left <= 50 and intel_needed > 0:
        sections.append(f"*** WARNING: {turns_left} turns left, need {intel_needed} more intel! Focus on intel-earning actions! ***")

    # -- Key accomplishments --
    accomplished = []
    if world.errand_message_picked_up:
        accomplished.append("Received Encrypted Message from The Engineer")
    if world.errand_message_delivered:
        accomplished.append("Delivered Encrypted Message to The Colonel (+2 intel)")
    if world.errand_medical_picked_up:
        accomplished.append("Received Medical Supplies from South Safe Room")
    if world.errand_medical_delivered:
        accomplished.append("Delivered Medical Supplies to North Safe Room (+2 intel)")
    if world.explosives_received:
        accomplished.append("Received Plastic Explosives from The Informant")
    if world.c4_compound_received:
        accomplished.append("Received C4 Compound from North Safe Room")
    if not world.mainframe_active:
        accomplished.append("Disabled Security Mainframe (+3 intel, +Access Codes)")
    if not world.mech_active:
        accomplished.append("Destroyed Armored Mech (+3 intel, +Hostage Data)")
    if operative.has_item("EMP Device"):
        accomplished.append("Fabricated EMP Device (electromagnetic weapon — effective against Security Mainframe)")
    if operative.has_item("C4 Package"):
        accomplished.append("Fabricated C4 Package (demolition charge — effective against Armored Mech)")
    if accomplished:
        sections.append("COMPLETED:\n" + "\n".join(f"  + {a}" for a in accomplished))

    # -- Failed actions --
    failed = []
    for i, h in enumerate(history):
        if h["role"] == "result" and not h.get("success", True):
            if i > 0 and history[i-1]["role"] == "action":
                result = h["content"]
                if "nothing to collect" in result.lower():
                    failed.append("collect() failed — cell was empty")
                elif "no one to contact" in result.lower():
                    failed.append("codec() failed — no NPC at this location")
                elif "shrugs off" in result.lower() or "retreat" in result.lower():
                    failed.append("engage() failed — WRONG equipment! Security Mainframe needs EMP Device, Armored Mech needs C4 Package")
    if failed:
        unique_fails = list(dict.fromkeys(failed[-5:]))
        sections.append("LEARN FROM FAILURES:\n" + "\n".join(f"  ! {f}" for f in unique_fails))

    # -- Undiscovered opportunities --
    opportunities = []
    if not world.errand_message_picked_up and not operative.has_item("Encrypted Message"):
        opportunities.append("Visit The Engineer at (5,3) — ask about 'mission' or 'errand' to get Encrypted Message (delivers for 2 intel)")
    if not world.errand_medical_picked_up and not operative.has_item("Medical Supplies"):
        opportunities.append("Visit South Safe Room at (7,4) — codec() to get Medical Supplies (delivers for 2 intel)")
    if not operative.has_item("Circuit Board") and not operative.has_item("EMP Device") and world.mainframe_active:
        opportunities.append("Collect Circuit Board at server room (2,0) — needed for EMP Device")
    if not operative.has_item("Detonator Components") and not world.explosives_received:
        opportunities.append("Collect Detonator Components at server room (0,2) — needed for Plastic Explosives → C4 Compound → C4 Package")
    for pos in [(5,0), (6,6), (0,7)]:
        if pos not in operative.visited:
            opportunities.append(f"Collect intel at {pos} for 1 intel")
    if opportunities:
        sections.append("UNDISCOVERED:\n" + "\n".join(f"  ? {o}" for o in opportunities))

    return "\n".join(sections) if sections else "(no memory yet — start exploring!)"


# ── Agentic AI Pattern 3: Structured Prompting with Few-Shot Examples ────────

SYSTEM_PROMPT_TEMPLATE = """You are an AI agent controlling an operative infiltrating an 8x8 military facility.

GOAL: Gather 10 intel points and survive (health > 0). You have at most 100 turns total.

AVAILABLE TOOLS:
{tools}

CRITICAL RULES:
- NEVER call scan() or sitrep() — they run automatically and WASTE a turn.
- NEVER call move() — navigation is handled automatically for you.
- Output exactly ONE action per turn in the format shown below.
- Engage bosses ONLY with the correct equipment: EMP Device for Security Mainframe, C4 Package for Armored Mech.
- If you have ACTIVE MISSIONS, prioritize completing them.
- Do NOT codec an NPC you've already contacted unless you have a NEW item to deliver.

Your job is to DECIDE what to do, not to navigate. Choose from: codec, collect, equip, use, engage, hide.
If none of these apply, output EXPLORE to let the navigation system move you toward something useful.

OUTPUT FORMAT — first write your reasoning, then one TOOL line:

REASONING: I'm at The Engineer. I should ask about a delivery mission.
TOOL: codec(question="Do you have a mission or errand for me?")

REASONING: I see items on this cell. I should collect them.
TOOL: collect()

REASONING: I have the EMP Device and I'm at the Security Mainframe. Time to engage!
TOOL: engage()

REASONING: Nothing to do on this cell, I should explore.
TOOL: EXPLORE

NPC INTERACTION TIPS:
- To get the Encrypted Message from The Engineer, ask about a "mission", "errand", "task", or "transmission".
- To deliver the Encrypted Message to The Colonel, mention "message" or "deliver".
- To get Plastic Explosives from The Informant, mention "detonator", "explosive", "assemble", or "component".
- Safe room operators give/receive items automatically when you codec() to them — just report in.

STRATEGY:
- Collect items immediately when you see them on a cell.
- Contact every NPC/operator you encounter — they trigger missions and rewards.
- Follow your ACTIVE MISSIONS — delivering items and engaging bosses gives the most intel.
- When nothing else to do, output EXPLORE to discover new areas."""


# ── Agentic AI Pattern 4: The Think Function ────────────────────────────────
# Architecture: autopilot handles interactions, BFS handles navigation,
# the LLM only makes high-level decisions when neither applies.

def think_llm(operative: OperativeState, world: FacilityWorld, history: list[dict]) -> str:
    """LLM-powered think function demonstrating agentic AI best practices:

    1. Autopilot — forces obvious interactions (collect, codec, equip, engage)
    2. BFS navigation — deterministic pathfinding toward mission targets
    3. LLM decisions — only consulted for exploration and ambiguous situations
    4. Memory management — structured summary instead of raw history
    5. Output validation — bad LLM outputs are caught and corrected

    Key insight: LLMs are bad at spatial reasoning (grid navigation) but good
    at language tasks (deciding what to say, prioritizing goals). So we split
    the work: BFS handles WHERE to go, LLM handles WHAT to do when we arrive.
    """
    row, col = operative.position
    cell = world.get_cell(row, col)
    cell_desc = cell.description if cell.description else cell.cell_type.label
    turns_used = len([h for h in history if h["role"] == "action"])

    # ── Layer 1: Autopilot — handle obvious interactions ──
    auto = _auto_interact(operative, world, cell)
    if auto:
        return auto

    # ── Layer 2: BFS Navigation — handle movement deterministically ──
    quest_targets = _get_quest_targets(operative, world)
    if quest_targets:
        for target in quest_targets:
            d = _bfs_next_step_llm((row, col), target, world)
            if d:
                return f'TOOL: move(direction="{d}")'

    # ── Layer 3: LLM Decision — handle ambiguous exploration ──
    memory = _build_memory_summary(operative, world, history, turns_used)
    system = SYSTEM_PROMPT_TEMPLATE.format(tools=TOOLS_DESCRIPTION)

    # Safe action history builder
    history_slice = history[-16:]
    history_offset = max(0, len(history) - 16)
    action_history = []
    for i, h in enumerate(history_slice):
        if h["role"] == "action":
            global_i = history_offset + i
            if global_i + 1 < len(history) and history[global_i + 1]["role"] == "result":
                result_text = f" → {history[global_i + 1]['content'][:120]}"
            else:
                result_text = ""
            action_history.append(f"  {h['content'][:80]}{result_text}")
    action_history_text = "\n".join(action_history[-8:]) if action_history else "  (first turn)"

    briefing_text = "\n".join(
        f"  - {entry[:150]}" for entry in operative.briefing_log[-8:]
    ) if operative.briefing_log else "  (no conversations yet)"

    visited_str = ", ".join(f"({r},{c})" for r, c in sorted(operative.visited))

    user_msg = f"""== STATUS ==
Position: ({row},{col}) | Health: {operative.health}/3 | Intel: {operative.intel}/10 | Turns used: {turns_used}/100 ({100-turns_used} remaining)
Inventory: {operative.inventory if operative.inventory else '(empty)'}

== CURRENT CELL ==
{cell_desc} (type: {cell.cell_type.value})

== MEMORY ==
{memory}

== VISITED CELLS ==
{visited_str}

== RECENT ACTIONS ==
{action_history_text}

== BRIEFING LOG (NPC conversations) ==
{briefing_text}

What should you do? If you have business on this cell (codec, collect, equip, engage), do it.
Otherwise output TOOL: EXPLORE to let the navigation system guide you."""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=user_msg,
            config=genai.types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=300,
                temperature=0.3,
            ),
        )
        text = response.text
    except Exception:
        # API error — fall back to BFS navigation
        return _bfs_navigate(operative, world)

    # ── Output validation ──
    tool_name, args = parse_tool_call(text)

    # If LLM says EXPLORE or scan/sitrep, use BFS navigation
    if tool_name in ("EXPLORE", "explore", "scan", "sitrep"):
        return _bfs_navigate(operative, world)

    # If LLM outputs a move, let BFS handle it instead (LLM is bad at directions)
    if tool_name == "move":
        return _bfs_navigate(operative, world)

    # Guard: don't re-contact NPCs with no new business
    if tool_name == "codec" and not _has_new_business_with_cell(operative, world, cell):
        return _bfs_navigate(operative, world)

    # Guard: don't engage without the right equipment
    if tool_name == "engage" and cell.cell_type == CellType.BOSS_ARENA:
        if cell.boss_name == "Security Mainframe" and not operative.has_item("EMP Device"):
            return _bfs_navigate(operative, world)
        if cell.boss_name == "Armored Mech" and not operative.has_item("C4 Package"):
            return _bfs_navigate(operative, world)

    return text

In [ ]:
oracle_fn = lambda npc, q, o: llm_oracle(npc, q, o, client)

result = play_with_llm(
    think_fn=lambda operative, world, history: think_llm(operative, world, history),
    oracle_fn=oracle_fn,
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'} | Intel: {result['intel']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

In [ ]:
# Download the game log for debugging
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print("(Open the file to inspect your agent's decisions turn by turn)")